# Universal Context Engine (Converged Edition)

**Copyright 2026, Denis Rothman**

**Goal:** In this notebook we will integrate **Oracle 23ai (Converged Enterprise DB):** agents in a universal content engine to prove that the exact same reasoning architecture (engine.py) can seamlessly switch between:

*  Oracle 23ai (Vector Store): For Marketing & Legal use cases.
*  Oracle 23ai (Relational Table): For HR & Recruitment use cases.


# 1.Installation & Setup

In [1]:
#@title Retrieving GitHub Private token (this cell will be deleted when the repository is public)
import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    pt = userdata.get("GITHUB_TOKEN")
    pt=pt.strip()
    if not pt:
        raise userdata.SecretNotFoundError("GITHUB_TOKEN not found.")

    print("GITHUB_TOKENAPI private token loaded successfully.")

except userdata.SecretNotFoundError:
    print('Secret "GITHUB_TOKEN" not found.')
    print('Please activate or add your GITHUB_TOKEN private token to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the GITHUB_TOKEN: {e}")

GITHUB_TOKENAPI private token loaded successfully.


In [2]:
# Install BOTH Pinecone and Oracle dependencies
!pip install oracledb==3.4.1 openai==2.14.0 tiktoken==0.7.0 tqdm==4.67.1 tenacity --quiet
print("✅ Universal Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.6 MB/s eta 0:00:00
✅ Universal Dependencies installed.


In [3]:
import os
import sys
from google.colab import drive, userdata
from openai import OpenAI

# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Initialize Oracle Client

try:
    # Explicitly check for the mandatory OpenAI key
    openai_key = userdata.get("API_KEY")
    if not openai_key:
        raise ValueError("OpenAI API_KEY not found in Secrets.")

    os.environ["OPENAI_API_KEY"] = openai_key
    client = OpenAI()
    print("✅ OpenAI initialized (Mandatory).")

except Exception as e:
    print(f"❌ Critical Error: OpenAI initialization failed. {e}")
    # We raise the error here because OpenAI is required for the Planner to work at all
    raise e

✅ OpenAI initialized (Mandatory).


In [5]:
#@title Retrieving engine library
import time

!curl -L -H "Authorization: token {pt}" -H "Accept: application/vnd.github.v3.raw" "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/commons/engine/engine.zip" -o engine.zip

time.sleep(10)
!unzip -o engine.zip -d /content

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 12940  100 12940    0     0  31920      0 --:--:-- --:--:-- --:--:-- 31950
Archive:  engine.zip
  inflating: /content/agent_archivist.py  
  inflating: /content/agent_recruiter.py  
  inflating: /content/agents.py      
  inflating: /content/engine.py      
  inflating: /content/helpers.py     
  inflating: /content/oracle_lib.py  
  inflating: /content/registry.py    


In [6]:
# 3. Import Local Modules (Uploaded Files)
try:
    from engine import context_engine
    from registry import AGENT_TOOLKIT
    from oracle_lib import OracleManager
    print("✅ Universal Engine Modules imported.")
except ImportError as e:
    print(f"❌ Module Import Failed: {e}\nUpload engine.py, registry.py, agents.py, helpers.py, oracle_lib.py, agent_archivist.py, agent_recruiter.py")

✅ Universal Engine Modules imported.


In [7]:
# Initialize Oracle Connection (Enterprise Layer)
try:
    OracleManager.initialize()
    print("✅ Oracle 23ai Connection established.")
except Exception as e:
    print(f"⚠️ Oracle Connection Failed: {e}\n(Ignore if you only want to run Pinecone scenarios)")

✅ Oracle 23ai Connection established.


# 2.Initialize the Universal Engine

In [12]:
# 2. Initialize the Universal Engine Wrapper
from engine import context_engine

# We define a wrapper function that pre-configures the engine with our clients and models.
# This allows us to just pass the 'goal' in the scenario steps below.
def run_universal_engine(goal):
    return context_engine(
        goal=goal,
        client=client,
        # [FIX] Removed Pinecone Client (pc), Index, and Namespaces
        generation_model="gpt-5.1",
        embedding_model="text-embedding-3-small"
    )

print("🚀 Universal Context Engine Wrapper is READY.")
# Access the registry dictionary keys directly to see the list of agents
print(list(AGENT_TOOLKIT.registry.keys()))

🚀 Universal Context Engine Wrapper is READY.
['Writer', 'Summarizer', 'OracleArchivist', 'OracleRecruiter']


# 3.Scenario B: Recruitment (Powered by Oracle)
This execution plan switches to the `OracleRecruiter` agent. The Engine doesn't change code, only the configuration.

In [13]:
#@title Oracle data verification
from oracle_lib import OracleManager

print("\n=== Oracle Hybrid Table Summary ===\n")

# Use the Manager's context manager instead of a raw 'connection' variable
with OracleManager.get_cursor() as cursor:
    # --- 1. Check HR Tables ---
    tables = ['CANDIDATE_POOL', 'RECRUITMENT_RULES']

    for t in tables:
        try:
            cursor.execute(f"SELECT COUNT(*) FROM {t}")
            result = cursor.fetchone()
            if result:
                print(f"Table {t}: {result[0]} rows")
        except Exception as e:
            print(f"Table {t}: Not Found or Error ({e})")

    print("\n--- Sample Candidate (Structured + Vector) ---")
    try:
        cursor.execute("""
            SELECT candidate_id, full_name, salary_expectation,
                   CASE WHEN resume_vector IS NOT NULL THEN '✅ Vector Present' ELSE '❌ No Vector' END
            FROM candidate_pool
            FETCH FIRST 2 ROWS ONLY
        """)
        for row in cursor.fetchall():
            print(row)
    except Exception as e:
        print(f"Error fetching candidates: {e}")

print("\n=== Verification Complete ===")


=== Oracle Hybrid Table Summary ===

Table CANDIDATE_POOL: 5 rows
Table RECRUITMENT_RULES: 3 rows

--- Sample Candidate (Structured + Vector) ---
('CAND_003', 'Casey M.', 210000, '✅ Vector Present')
('CAND_001', 'Alex V.', 165000, '✅ Vector Present')

=== Verification Complete ===


In [14]:
#@title Query powered by Oracle
# 4. Scenario B: Recruitment (Powered by Oracle)
# This execution plan switches to the `OracleRecruiter` agent.
# The Engine doesn't change code, only the configuration.

# We state the constraints clearly in the goal so the Planner passes them to the OracleRecruiter.
recruitment_goal = "Find Experienced Python Developers with max salary 140k and min 3 years experience, then draft a polite rejection email for them mentioning we will keep their resume."

print(f"Goal: {recruitment_goal}")

# Execute the engine (The Planner will automatically select OracleRecruiter -> Writer)
result, trace = run_universal_engine(recruitment_goal)

print("\n--- FINAL RECRUITMENT EMAILS ---")
print(result)

Goal: Find Experienced Python Developers with max salary 140k and min 3 years experience, then draft a polite rejection email for them mentioning we will keep their resume.

--- FINAL RECRUITMENT EMAILS ---
Subject: Update on Your Application

Hi [Candidate Name],

Thank you very much for your interest in the [Role Title] position at [Company Name] and for the time you invested in our interview process.

After careful consideration, we’ve decided to move forward with another candidate for this particular role. This was a difficult decision given the strength of your background and experience.

We will keep your resume on file and may reach out if future opportunities arise that align more closely with your skills and career goals.

We appreciate your interest in [Company Name] and wish you all the best in your continued search. Please don’t hesitate to stay in touch or reapply for roles that interest you.

Best regards,  
[Your Name]  
[Your Title]  
[Company Name]  
[Contact Informati

In [15]:
# 4. Scenario B: Recruitment (Powered by Oracle)
# This execution plan switches to the `OracleRecruiter` agent.
# The Engine doesn't change code, only the configuration.

# We state the constraints clearly in the goal so the Planner passes them to the OracleRecruiter.
# UPGRADE: Explicitly used "140000" (integer format) to ensure Planner parsing accuracy.
recruitment_goal = "Find Experienced Python Developers with max salary 140000 and min 3 years experience, then draft a polite rejection email for them mentioning we will keep their resume."

print(f"Goal: {recruitment_goal}")

# Execute the engine (The Planner will automatically select OracleRecruiter -> Writer)
result, trace = run_universal_engine(recruitment_goal)

# --- RESULT HANDLING & DEBUGGING ---
if result:
    print("\n--- FINAL RECRUITMENT EMAILS ---")
    print(result)
else:
    print("\n⚠️ No Final Result returned by Engine. Inspecting Trace...")
    print(f"Trace Status: {trace.status}")

    # 1. Did we get a plan?
    if trace.plan:
        print(f"Plan Generated: {len(trace.plan)} steps")
        for step in trace.plan:
            print(f" - Step {step.get('step')}: {step.get('agent')} -> {step.get('input')}")
    else:
        print("❌ CRITICAL: No Plan was generated. Check your LLM / Planner logic.")

    # 2. Did we execute steps?
    if trace.steps:
        print(f"\nSteps Executed: {len(trace.steps)}")
        last_step = trace.steps[-1]
        print(f"Last Agent: {last_step['agent']}")
        print(f"Last Output Snippet: {str(last_step['output'])[:200]}...")
    else:
        print("❌ CRITICAL: Plan generated but no steps executed.")

Goal: Find Experienced Python Developers with max salary 140000 and min 3 years experience, then draft a polite rejection email for them mentioning we will keep their resume.

--- FINAL RECRUITMENT EMAILS ---
Subject: Application Update

Dear [Candidate Name],

Thank you again for taking the time to apply for the Python Developer position and for your interest in joining our team.

After careful consideration, we have decided not to move forward with your application at this time. This was a difficult decision given the strength of your background and experience.

We will keep your resume on file and may reach out should a future opportunity arise that aligns more closely with your skills and experience.

We truly appreciate the effort you put into your application and wish you all the best in your ongoing job search and future endeavors.

Sincerely,  
[Your Name]  
[Your Title]  
[Company Name]  
[Contact Information]


In [ ]:
OracleManager.close()